# 28 — Transports and Capability Negotiation, Revisited

This closing notebook of Part 3 ties the protocol internals from notebooks
24-27 back to the transports you've actually used across the whole course:
stdio (Parts 1 and 3) and Streamable HTTP (Part 2's Azure API Management-
hosted servers). Read alongside
[docs/12_mcp_protocol_foundations.md](../docs/12_mcp_protocol_foundations.md).

## stdio, precisely

- The client launches the server as a **subprocess**.
- Messages are newline-delimited JSON-RPC on `stdin`/`stdout`; a message
  **MUST NOT** contain an embedded newline.
- `stderr` is free for logs -- the server **MUST NOT** write anything to
  `stdout` that isn't a valid MCP message.
- Shutdown: the client closes the server's `stdin`, waits, then escalates
  to `SIGTERM`/`SIGKILL` if needed.

`examples/mcp_foundations/raw_jsonrpc_client.py` (notebook 24) implements
exactly these rules with nothing but `subprocess` and `json` -- re-read it
now that you've seen the full protocol it's a minimal implementation of.

## Streamable HTTP, precisely

- A **single HTTP endpoint** (conventionally `/mcp`) handles both `POST`
  (client → server messages) and `GET` (optional server → client SSE
  stream).
- Every client-sent JSON-RPC message is its own `POST`; the response is
  either one JSON object or an SSE stream of messages ending in the
  matching response.
- Clients **MUST** send `Accept: application/json, text/event-stream`;
  servers **MUST** validate the `Origin` header to prevent DNS rebinding.
- After the first successful response, the client **MUST** send
  `MCP-Protocol-Version: <version>` on every subsequent request.

This is exactly the transport Azure API Management uses for every MCP
server you created in notebooks 17 and 20 -- "AI Gateway" didn't invent a
new transport, it's a governed front door onto this one.

## Capability negotiation, mapped onto this course

| Server | `tools` | `resources` | `prompts` | Notes |
| --- | --- | --- | --- | --- |
| `examples/mcp_foundations/custom_server.py` (notebooks 25-27) | ✅ | ✅ | ✅ | Everything the spec offers, stdio only |
| Azure MCP Server (Part 1) | ✅ | ❌ | ❌ | Tools-only by design -- see `docs/06_available_tools_catalog.md` |
| API Management REST-API-backed MCP server (notebook 17) | ✅ | ❌ | ❌ | API operations become tools; resources/prompts unsupported |
| API Management passthrough MCP server (notebook 20) | ✅ | ✅ | ❌ | Can proxy an external server's resources too; prompts still unsupported |

Knowing *why* a server's capability set looks the way it does -- a
deliberate design choice vs. a platform limitation -- is the kind of
judgment Part 3 was meant to give you.

## Course recap

- **Part 1 (01-15):** use the Azure MCP Server as an MCP client -- tools,
  function calling, multi-turn agents, security.
- **Part 2 (16-23):** govern MCP servers and LLM backends through Azure API
  Management's AI gateway, and decide when its AI-specific policies
  actually apply.
- **Part 3 (24-28):** understand the protocol itself well enough to read
  or build any MCP server, independent of Azure entirely.

From here: build something. Wrap an internal tool as an MCP server
(notebook 25's pattern), expose it through API Management if it needs to
be shared and governed (notebook 17), and let any MCP-compatible agent use
it.

## Further reading

- [Model Context Protocol specification](https://modelcontextprotocol.io/specification/2025-06-18)
- [MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk) --
  `docs/server.md` and `docs/client.md` cover every primitive in this
  notebook series in more depth than fits here.
- [docs/01_what_is_mcp.md](../docs/01_what_is_mcp.md) -- where this course
  started; worth rereading now that Part 3 has filled in the details.